#### ***Parameters***
---

In [ ]:
import numpy as np
import pandas as pd

PLANT_ID = 4135001
START_DATE = "2020-05-15"
END_DATE = "2020-05-31"
START_HOUR, END_HOUR = 8, 17
IRRADIATION_THRESHOLD = 0.3

---
#### ***Data Ingestion***

In [30]:
generation_data = pd.read_csv("data/Plant_1_Generation_Data.csv")
weather_data = pd.read_csv("data/Plant_1_Weather_Sensor_Data.csv")

print(f"  Generation rows : {len(generation_data)}")
print(f"  Weather rows    : {len(weather_data)}")

  Generation rows : 68778
  Weather rows    : 3182


---
#### ***Unique Filter***

In [ ]:
# Filter by plant ID
gen_plant = generation_data[generation_data["PLANT_ID"] == PLANT_ID].copy()
wth_plant = weather_data[weather_data["PLANT_ID"] == PLANT_ID].copy()

print(gen_plant[["PLANT_ID", "DATE_TIME", "AC_POWER", "DC_POWER"]].head(5))

   PLANT_ID         DATE_TIME  AC_POWER  DC_POWER
0   4135001  15-05-2020 00:00       0.0       0.0
1   4135001  15-05-2020 00:00       0.0       0.0
2   4135001  15-05-2020 00:00       0.0       0.0
3   4135001  15-05-2020 00:00       0.0       0.0
4   4135001  15-05-2020 00:00       0.0       0.0


In [ ]:
# Parse DATE_TIME > DATETIME and Extract HOUR
gen_plant["DATETIME"] = pd.to_datetime(gen_plant["DATE_TIME"], format="%d-%m-%Y %H:%M")
wth_plant["DATETIME"] = pd.to_datetime(wth_plant["DATE_TIME"])
gen_plant["HOUR"] = gen_plant["DATETIME"].dt.hour
wth_plant["HOUR"] = wth_plant["DATETIME"].dt.hour

print("Parse DATE_TIME > DATETIME and Extract HOUR")
print("─" * 50)
print(gen_plant[["PLANT_ID", "DATETIME", "HOUR", "AC_POWER", "DC_POWER"]].head(5))

# Filter by date range
gen_dated = gen_plant[
    (gen_plant["DATETIME"] >= START_DATE) & (gen_plant["DATETIME"] <= END_DATE)
].copy()
wth_dated = wth_plant[
    (wth_plant["DATETIME"] >= START_DATE) & (wth_plant["DATETIME"] <= END_DATE)
].copy()

print("─" * 50)
print("Filter by date range")
print("─" * 50)
print(gen_dated[["PLANT_ID", "DATETIME", "HOUR", "AC_POWER"]].head(5))

# Filter by daytime hours
gen_day = gen_dated[
    (gen_dated["HOUR"] >= START_HOUR) & (gen_dated["HOUR"] <= END_HOUR)
].copy()
wth_day = wth_dated[
    (wth_dated["HOUR"] >= START_HOUR) & (wth_dated["HOUR"] <= END_HOUR)
].copy()

print("─" * 50)
print("Filter bt daytime hours")
print("─" * 50)
print(gen_day[["PLANT_ID", "DATETIME", "HOUR", "AC_POWER"]].head(5))

Parse DATE_TIME > DATETIME and Extract HOUR
──────────────────────────────────────────────────
   PLANT_ID   DATETIME  HOUR  AC_POWER  DC_POWER
0   4135001 2020-05-15     0       0.0       0.0
1   4135001 2020-05-15     0       0.0       0.0
2   4135001 2020-05-15     0       0.0       0.0
3   4135001 2020-05-15     0       0.0       0.0
4   4135001 2020-05-15     0       0.0       0.0
──────────────────────────────────────────────────
Filter by date range
──────────────────────────────────────────────────
   PLANT_ID   DATETIME  HOUR  AC_POWER
0   4135001 2020-05-15     0       0.0
1   4135001 2020-05-15     0       0.0
2   4135001 2020-05-15     0       0.0
3   4135001 2020-05-15     0       0.0
4   4135001 2020-05-15     0       0.0
──────────────────────────────────────────────────
Filter bt daytime hours
──────────────────────────────────────────────────
     PLANT_ID            DATETIME  HOUR    AC_POWER
684   4135001 2020-05-15 08:00:00     8  318.671429
685   4135001 2020-05-15

---
#### ***Data Cleaning***

In [ ]:
# Before cleaning
gen_before = len(gen_day)
wth_before = len(wth_day)
gen_missing_before = gen_day.isnull().sum().sum()
wth_missing_before = wth_day.isnull().sum().sum()

print("Before Cleaning")
print("─" * 50)
print(f"  Generation — rows: {gen_before}, missing values: {gen_missing_before}")
print(f"  Weather    — rows: {wth_before}, missing values: {wth_missing_before}")

# Remove duplicates
gen_day = gen_day.drop_duplicates()
wth_day = wth_day.drop_duplicates()

gen_dupes_removed = gen_before - len(gen_day)
wth_dupes_removed = wth_before - len(wth_day)

# Median imputation for missing numeric values
for col in gen_day.select_dtypes(include="number").columns:
    gen_day[col] = gen_day[col].fillna(gen_day[col].median())
for col in wth_day.select_dtypes(include="number").columns:
    wth_day[col] = wth_day[col].fillna(wth_day[col].median())

print("─" * 50)
print("After Cleaning")
print("─" * 50)
print(
    f"  Generation — rows: {len(gen_day)}, duplicates removed: {gen_dupes_removed}, missing values: {gen_day.isnull().sum().sum()}"
)
print(
    f"  Weather    — rows: {len(wth_day)}, duplicates removed: {wth_dupes_removed}, missing values: {wth_day.isnull().sum().sum()}"
)
print("─" * 50)
print("Missing values per column (generation):")
print(gen_day.isnull().sum().rename("missing"))
print("─" * 50)
print("Missing values per column (weather):")
print(wth_day.isnull().sum().rename("missing"))

Before Cleaning
──────────────────────────────────────────────────
  Generation — rows: 13638, missing values: 0
  Weather    — rows: 621, missing values: 0
──────────────────────────────────────────────────
After Cleaning
──────────────────────────────────────────────────
  Generation — rows: 13638, duplicates removed: 0, missing values: 0
  Weather    — rows: 621, duplicates removed: 0, missing values: 0
──────────────────────────────────────────────────
Missing values per column (generation):
DATE_TIME      0
PLANT_ID       0
SOURCE_KEY     0
DC_POWER       0
AC_POWER       0
DAILY_YIELD    0
TOTAL_YIELD    0
DATETIME       0
HOUR           0
Name: missing, dtype: int64
──────────────────────────────────────────────────
Missing values per column (weather):
DATE_TIME              0
PLANT_ID               0
SOURCE_KEY             0
AMBIENT_TEMPERATURE    0
MODULE_TEMPERATURE     0
IRRADIATION            0
DATETIME               0
HOUR                   0
Name: missing, dtype: int64


---
#### ***Data merging***

In [ ]:
gen_agg = (
    gen_day.groupby(["DATETIME", "PLANT_ID"])
    .agg(
        {
            "DC_POWER": "sum",
            "AC_POWER": "sum",
            "DAILY_YIELD": "mean",
            "TOTAL_YIELD": "mean",
        }
    )
    .reset_index()
)
merged_data = pd.merge(gen_agg, wth_day, on=["DATETIME", "PLANT_ID"], how="inner")
merged_data = merged_data.drop(
    columns=["DATE_TIME_x", "DATE_TIME_y", "HOUR_x", "HOUR_y"], errors="ignore"
)

print(
    merged_data[
        ["DATETIME", "AC_POWER", "DC_POWER", "IRRADIATION", "MODULE_TEMPERATURE"]
    ].head(5)
)

             DATETIME      AC_POWER       DC_POWER  IRRADIATION  \
0 2020-05-15 08:00:00   6365.608928   64834.803572     0.201639   
1 2020-05-15 08:15:00  10002.064643  101937.157142     0.345708   
2 2020-05-15 08:30:00  10995.602976  112129.565473     0.405349   
3 2020-05-15 08:45:00   9080.067857   92505.464285     0.312427   
4 2020-05-15 09:00:00  16211.350000  165598.892858     0.623153   

   MODULE_TEMPERATURE  
0           31.412545  
1           35.528711  
2           40.318059  
3           39.081954  
4           45.009233  


---
#### ***Classification***

In [42]:
# Irradiation threshold + Efficiency ratio
merged_data["CLOUD_CONDITION"] = np.where(
    merged_data["IRRADIATION"] < IRRADIATION_THRESHOLD, "Cloud-Affected", "Clear-Sky"
)

merged_data["EFFICIENCY_RATIO"] = np.where(
    merged_data["DC_POWER"] > 0, merged_data["AC_POWER"] / merged_data["DC_POWER"], 0
)

cloud_df = merged_data[merged_data["CLOUD_CONDITION"] == "Cloud-Affected"]
clear_df = merged_data[merged_data["CLOUD_CONDITION"] == "Clear-Sky"]

print("Classification")
print("─" * 50)
print(f"  Irradiation threshold : {IRRADIATION_THRESHOLD} kW/m²")
print(f"  Cloud-Affected        : {len(cloud_df)} rows")
print(f"  Clear-Sky             : {len(clear_df)} rows")
print("─" * 50)
print(
    merged_data[
        [
            "DATETIME",
            "IRRADIATION",
            "AC_POWER",
            "DC_POWER",
            "EFFICIENCY_RATIO",
            "CLOUD_CONDITION",
        ]
    ].head(5)
)

Classification
──────────────────────────────────────────────────
  Irradiation threshold : 0.3 kW/m²
  Cloud-Affected        : 126 rows
  Clear-Sky             : 495 rows
──────────────────────────────────────────────────
             DATETIME  IRRADIATION      AC_POWER       DC_POWER  \
0 2020-05-15 08:00:00     0.201639   6365.608928   64834.803572   
1 2020-05-15 08:15:00     0.345708  10002.064643  101937.157142   
2 2020-05-15 08:30:00     0.405349  10995.602976  112129.565473   
3 2020-05-15 08:45:00     0.312427   9080.067857   92505.464285   
4 2020-05-15 09:00:00     0.623153  16211.350000  165598.892858   

   EFFICIENCY_RATIO CLOUD_CONDITION  
0          0.098182  Cloud-Affected  
1          0.098120       Clear-Sky  
2          0.098062       Clear-Sky  
3          0.098157       Clear-Sky  
4          0.097895       Clear-Sky  


---
<span style="font-size: 8px;">R01001100</span>